# 07 – Merge de fuentes

Este notebook integra todas las fuentes intermedias disponibles para construir el dataset final de entrenamiento.

## Fuentes consideradas

- `milking.parquet`
- `rumination.parquet`
- `weather.parquet`
- `events.parquet`
- `feeding_daily_corral.parquet`
- `diet_period_summary.parquet`

## Objetivos

- cargar y normalizar todas las fuentes
- revisar cobertura temporal y llaves de merge
- construir datasets diarios consistentes
- hacer merge principal sobre `cow_id + date`
- incorporar alimentación y dieta
- exportar `training_dataset.parquet`

## Nota importante sobre alimentación

Los datos de alimentación están a nivel:

```text
date + corral_id
```

y no a nivel:

```text
cow_id + date
```

Como en esta etapa no existe un mapeo confiable `cow_id -> corral_id` por fecha, la alimentación se agregará primero a nivel diario global y luego se unirá por `date`.


## 1. Importaciones


In [22]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)


## 2. Rutas


In [23]:
BASE_INTERIM = Path("../data/interim")
BASE_PROCESSED = Path("../data/processed")

PATH_MILKING = BASE_INTERIM / "milking.parquet"
PATH_RUMINATION = BASE_INTERIM / "rumination.parquet"
PATH_WEATHER = BASE_INTERIM / "weather.parquet"
PATH_EVENTS = BASE_INTERIM / "events.parquet"
PATH_FEEDING = BASE_INTERIM / "feeding_daily_corral.parquet"
PATH_DIET = BASE_INTERIM / "diet_period_summary.parquet"

OUT_TRAINING = BASE_PROCESSED / "training_dataset.parquet"

print(PATH_MILKING)
print(PATH_RUMINATION)
print(PATH_WEATHER)
print(PATH_EVENTS)
print(PATH_FEEDING)
print(PATH_DIET)


../data/interim/milking.parquet
../data/interim/rumination.parquet
../data/interim/weather.parquet
../data/interim/events.parquet
../data/interim/feeding_daily_corral.parquet
../data/interim/diet_period_summary.parquet


## 3. Carga de todas las fuentes


In [24]:
milking_raw = pd.read_parquet(PATH_MILKING)
rumination_raw = pd.read_parquet(PATH_RUMINATION)
weather_raw = pd.read_parquet(PATH_WEATHER)
events_raw = pd.read_parquet(PATH_EVENTS)
feeding_raw = pd.read_parquet(PATH_FEEDING)
diet_raw = pd.read_parquet(PATH_DIET)

print("milking_raw     ->", milking_raw.shape)
print("rumination_raw  ->", rumination_raw.shape)
print("weather_raw     ->", weather_raw.shape)
print("events_raw      ->", events_raw.shape)
print("feeding_raw     ->", feeding_raw.shape)
print("diet_raw        ->", diet_raw.shape)

display(milking_raw.head())
display(rumination_raw.head())
display(weather_raw.head())
display(events_raw.head())
display(feeding_raw.head())
display(diet_raw.head())


milking_raw     -> (23763, 17)
rumination_raw  -> (19110, 15)
weather_raw     -> (29376, 7)
events_raw      -> (6890, 6)
feeding_raw     -> (1638, 13)
diet_raw        -> (3, 15)


,numero_ordeno,duracion_mmss,produccion_kg,intervalo_ordeno_hhmm,di,dd,ti,td,ubre,pezon,destino_leche,ms,cow_id,source_file,duracion_min,intervalo_ordeno_min,date
0,1,11:27,18.99,21:01,5.49,5.39,2.27,5.84,1,NaN,Tanque,VMS 1,1213,Producciones de leche 1213.xls,NaN,NaN,2025-01-01
1,2,05:44,13.60,13:14,3.82,4.03,1.35,4.40,1,NaN,Tanque,VMS 1,1213,Producciones de leche 1213.xls,NaN,NaN,2025-01-01
2,1,07:04,12.58,10:25,3.66,3.62,1.41,3.89,0,NaN,Tanque,VMS 1,1213,Producciones de leche 1213.xls,NaN,NaN,2025-01-02
3,1,11:18,19.38,24:51,5.67,6.06,2.02,5.63,0,NaN,Tanque,VMS 1,1213,Producciones de leche 1213.xls,NaN,NaN,2025-01-03
4,2,09:12,18.90,17:16,5.23,5.61,2.09,5.97,0,NaN,Tanque,VMS 1,1213,Producciones de leche 1213.xls,NaN,NaN,2025-01-03


,cow_id,eid,lid,group_id,last_calving,days_in_milk,date,ruminating_minutes,source_file,group_from_file,collar_from_file,weekday,month,lactation_age,resolved_group
0,1243,NaN,NaN,20.0,2024-08-29,302.0,2025-06-27,NaN,group_100_ruminating_rumia.csv,100.0,NaN,4,6,302,20.0
1,1243,NaN,NaN,20.0,2024-08-29,303.0,2025-06-28,553.0,group_100_ruminating_rumia.csv,100.0,NaN,5,6,303,20.0
2,1243,NaN,NaN,20.0,2024-08-29,304.0,2025-06-29,611.0,group_100_ruminating_rumia.csv,100.0,NaN,6,6,304,20.0
3,1243,NaN,NaN,20.0,2024-08-29,305.0,2025-06-30,646.0,group_100_ruminating_rumia.csv,100.0,NaN,0,6,305,20.0
4,1243,NaN,NaN,20.0,2024-08-29,306.0,2025-07-01,536.0,group_100_ruminating_rumia.csv,100.0,NaN,1,7,306,20.0


,date,time,temperature_2m,relative_humidity_2m,pressure_msl,precipitation,wind_speed_10m
0,2022-05-26,2022-05-26 00:00:00,16.8,83,1017.0,0.0,13.0
1,2022-05-26,2022-05-26 01:00:00,16.0,87,1016.8,0.0,13.0
2,2022-05-26,2022-05-26 02:00:00,15.4,91,1016.7,0.0,13.3
3,2022-05-26,2022-05-26 03:00:00,14.8,96,1016.9,0.0,12.2
4,2022-05-26,2022-05-26 04:00:00,14.4,98,1017.3,0.0,9.5


,cow_id,page_number,lactation_number,raw_event_text,source_file,date
0,1204,1,1,Cambio tabla ali 22/08/2 User1,Eventos de animales 1204.pdf,2022-08-22
1,1204,1,1,Condición corporal: 3.25 - DDUP:,Eventos de animales 1204.pdf,NaT
2,1204,1,1,Condición corpo 22/08/2 473 User1 Dry Off SE S...,Eventos de animales 1204.pdf,2022-08-22
3,1204,1,1,Secado 22/08/2 User1 GESTANTE EN LA 7 INSEM. S...,Eventos de animales 1204.pdf,2022-08-22
4,1204,1,1,Cambio de grup 22/08/2 User1 Dns: VACUNA; Loc....,Eventos de animales 1204.pdf,2022-08-22


,date,corral_id,kg_am,kg_pm,kg_totales,sobrante,consumo,source_sheet,rechazo,kg_consumido_por_vaca,promedio_corral,sobrante_pct,consumo_pct
0,2025-01-01,1.0,500.0,500.0,1000.0,0.0,1000.0,Enero,NaN,NaN,NaN,0.000000,1.000000
1,2025-01-01,2.0,650.0,650.0,1300.0,0.0,1300.0,Enero,NaN,NaN,NaN,0.000000,1.000000
2,2025-01-01,3.0,600.0,600.0,1200.0,0.0,1200.0,Enero,NaN,NaN,NaN,0.000000,1.000000
3,2025-01-01,4.0,650.0,650.0,1300.0,10.0,1290.0,Enero,NaN,NaN,NaN,0.007692,0.992308
4,2025-01-01,5.0,600.0,600.0,1200.0,0.0,1200.0,Enero,NaN,NaN,NaN,0.000000,1.000000


,diet_period,period_start,period_end,n_ingredientes,diet_dm_total,diet_wet_total,diet_pct_ms_total,usa_oro_milk,usa_oro_balance,usa_silo_maiz,usa_silo_avena,usa_ensilado,usa_heno,usa_triticale,usa_melaza
0,dieta Mayo,2025-01-01,2025-05-31,5,NaN,NaN,NaN,1,0,0,0,0,0,1,1
1,Dieta Junio,2025-06-01,2025-07-31,6,NaN,NaN,NaN,1,1,0,0,0,0,1,1
2,Dieta Agosto,2025-08-01,2025-09-30,9,NaN,NaN,NaN,1,1,0,1,0,0,0,1


### Comentarios y observaciones

Aquí conviene verificar:

- que todas las fuentes realmente existen
- que las columnas clave estén presentes
- que las fechas se puedan normalizar correctamente
- que no haya tablas vacías


## 4. Normalización de fechas y tipos


In [25]:
print(weather_raw.columns.tolist())
display(weather_raw.head())

['date', 'time', 'temperature_2m', 'relative_humidity_2m', 'pressure_msl', 'precipitation', 'wind_speed_10m']


,date,time,temperature_2m,relative_humidity_2m,pressure_msl,precipitation,wind_speed_10m
0,2022-05-26,2022-05-26 00:00:00,16.8,83,1017.0,0.0,13.0
1,2022-05-26,2022-05-26 01:00:00,16.0,87,1016.8,0.0,13.0
2,2022-05-26,2022-05-26 02:00:00,15.4,91,1016.7,0.0,13.3
3,2022-05-26,2022-05-26 03:00:00,14.8,96,1016.9,0.0,12.2
4,2022-05-26,2022-05-26 04:00:00,14.4,98,1017.3,0.0,9.5


In [26]:
def normalize_date_col(df, col="date"):
    df = df.copy()
    df[col] = pd.to_datetime(df[col], errors="coerce").dt.normalize()
    return df

milking_raw = normalize_date_col(milking_raw, "date")
rumination_raw = normalize_date_col(rumination_raw, "date")
weather_raw = normalize_date_col(weather_raw, "date")
events_raw = normalize_date_col(events_raw, "date")
feeding_raw = normalize_date_col(feeding_raw, "date")

if "period_start" in diet_raw.columns:
    diet_raw["period_start"] = pd.to_datetime(diet_raw["period_start"], errors="coerce").dt.normalize()
if "period_end" in diet_raw.columns:
    diet_raw["period_end"] = pd.to_datetime(diet_raw["period_end"], errors="coerce").dt.normalize()

for df_name, df in {
    "milking_raw": milking_raw,
    "rumination_raw": rumination_raw,
    "weather_raw": weather_raw,
    "events_raw": events_raw
}.items():
    if "cow_id" in df.columns:
        df["cow_id"] = pd.to_numeric(df["cow_id"], errors="coerce").astype("Int64")
    if df_name == "milking_raw":
        milking_raw = df
    elif df_name == "rumination_raw":
        rumination_raw = df
    elif df_name == "weather_raw":
        weather_raw = df
    elif df_name == "events_raw":
        events_raw = df

print(milking_raw.dtypes.head(10))
print(rumination_raw.dtypes.head(10))
print(weather_raw.dtypes.head(10))
print(events_raw.dtypes.head(10))


numero_ordeno              int64
duracion_mmss                str
produccion_kg            float64
intervalo_ordeno_hhmm        str
di                       float64
dd                       float64
ti                       float64
td                       float64
ubre                       int64
pezon                        str
dtype: object
cow_id                         Int64
eid                          float64
lid                          float64
group_id                     float64
last_calving          datetime64[us]
days_in_milk                 float64
date                  datetime64[us]
ruminating_minutes           float64
source_file                      str
group_from_file              float64
dtype: object
date                     datetime64[s]
time                    datetime64[us]
temperature_2m                 float64
relative_humidity_2m             int64
pressure_msl                   float64
precipitation                  float64
wind_speed_10m                 float64

## 5. Construcción de tablas diarias


### 5.1 Milking diario

Esta fuente ya suele venir a nivel vaca-fecha, pero aquí se vuelve a agregar por seguridad.


In [27]:
milking_agg = {}
for c in ["produccion_kg", "di", "dd", "ti", "td", "ordenos_dia", "duracion_total_min"]:
    if c in milking_raw.columns:
        milking_agg[c] = "sum"

for c in ["intervalo_ordeno_prom_min"]:
    if c in milking_raw.columns:
        milking_agg[c] = "mean"

for c in ["ubre", "destino_leche", "ms", "source_file"]:
    if c in milking_raw.columns:
        milking_agg[c] = "first"

milking_daily = (
    milking_raw
    .dropna(subset=["cow_id", "date"])
    .groupby(["cow_id", "date"], as_index=False)
    .agg(milking_agg)
)

print("milking_daily ->", milking_daily.shape)
display(milking_daily.head())


milking_daily -> (9814, 11)


,cow_id,date,produccion_kg,di,dd,ti,td,ubre,destino_leche,ms,source_file
0,1204,2025-01-01,16.68,5.16,4.64,0.00,6.88,0,Tanque,VMS 1,Producciones de leche1204.xls
1,1204,2025-01-02,17.91,2.65,4.44,5.75,5.07,1,Divert 3,VMS 1,Producciones de leche1204.xls
2,1204,2025-01-03,25.36,6.26,6.52,2.77,9.81,1,Tanque,VMS 1,Producciones de leche1204.xls
3,1204,2025-01-04,16.71,4.13,3.89,2.96,5.73,1,Tanque,VMS 1,Producciones de leche1204.xls
4,1204,2025-01-05,25.04,5.85,5.94,4.85,8.40,1,Tanque,VMS 1,Producciones de leche1204.xls


### 5.2 Rumination diaria

La fuente de rumia también se agrega por vaca y fecha.  
Se usan sumas o primeros valores según corresponda.


In [28]:
rum_agg = {}
for c in ["rumia_min"]:
    if c in rumination_raw.columns:
        rum_agg[c] = "sum"

for c in ["days_in_milk", "lactation_age", "weekday", "month", "group_id", "source_file"]:
    if c in rumination_raw.columns:
        rum_agg[c] = "first"

rumination_daily = (
    rumination_raw
    .dropna(subset=["cow_id", "date"])
    .groupby(["cow_id", "date"], as_index=False)
    .agg(rum_agg)
)

print("rumination_daily ->", rumination_daily.shape)
display(rumination_daily.head())


rumination_daily -> (7098, 8)


,cow_id,date,days_in_milk,lactation_age,weekday,month,group_id,source_file
0,1204,2025-06-27,417.0,417,4,6,100.0,group_100_ruminating_rumia.csv
1,1204,2025-06-28,418.0,418,5,6,100.0,group_100_ruminating_rumia.csv
2,1204,2025-06-29,419.0,419,6,6,100.0,group_100_ruminating_rumia.csv
3,1204,2025-06-30,420.0,420,0,6,100.0,group_100_ruminating_rumia.csv
4,1204,2025-07-01,421.0,421,1,7,100.0,group_100_ruminating_rumia.csv


### 5.3 Clima diario

El clima se unirá solo por fecha.


In [29]:
weather_agg = {}
for c in ["temperature_2m", "relative_humidity_2m"]:
    if c in weather_raw.columns:
        weather_agg[c] = "mean"

weather_daily = (
    weather_raw
    .dropna(subset=["date"])
    .groupby("date", as_index=False)
    .agg(weather_agg)
)

print("weather_daily ->", weather_daily.shape)
display(weather_daily.head())


weather_daily -> (1224, 3)


,date,temperature_2m,relative_humidity_2m
0,2022-05-26,20.433333,63.458333
1,2022-05-27,19.979167,61.000000
2,2022-05-28,19.337500,56.875000
3,2022-05-29,20.716667,48.708333
4,2022-05-30,21.520833,49.166667


### 5.4 Eventos diarios

Los eventos se agregan por vaca y fecha.


In [30]:
if "eventos_pdf_text" not in events_raw.columns:
    text_col = [c for c in events_raw.columns if "text" in c.lower()]
    if text_col:
        events_raw = events_raw.rename(columns={text_col[0]: "eventos_pdf_text"})

events_daily = (
    events_raw
    .dropna(subset=["cow_id", "date"])
    .groupby(["cow_id", "date"], as_index=False)
    .agg(
        eventos_pdf_count=("cow_id", "size"),
        eventos_pdf_text=("eventos_pdf_text", lambda x: " | ".join([str(v) for v in x.dropna().unique()]))
    )
)

print("events_daily ->", events_daily.shape)
display(events_daily.head())


events_daily -> (3822, 4)


,cow_id,date,eventos_pdf_count,eventos_pdf_text
0,1204,2022-01-04,1,Control de Gest 04/01/2 User1
1,1204,2022-01-09,2,Diagnósticos/Tr 09/01/2 Med: User1 - VACIA DE ...
2,1204,2022-01-17,2,Inseminación 17/01/2 fertility/ANGUS; 2406261;...
3,1204,2022-01-21,1,Cambio de grup 21/01/2 User1 Artificial Insemi...
4,1204,2022-02-20,2,Diagnósticos/Tr 20/02/2 OVS; Med: User1 - VACI...


### 5.5 Alimentación diaria agregada

Como aún no se tiene el corral por vaca por fecha, se agrega la alimentación a nivel global por día.


In [31]:
feeding_agg = {}
for c in [
    "kg_am", "kg_pm", "kg_totales", "sobrante", "consumo",
    "rechazo", "n_vacas", "kg_consumido_por_vaca",
    "promedio_corral", "kg_ofrecidos_por_vaca",
    "sobrante_pct", "consumo_pct"
]:
    if c in feeding_raw.columns:
        # sumas para cantidades totales, medias para razones y promedios
        if c in ["kg_am", "kg_pm", "kg_totales", "sobrante", "consumo", "n_vacas"]:
            feeding_agg[c] = "sum"
        else:
            feeding_agg[c] = "mean"

feeding_daily = (
    feeding_raw
    .dropna(subset=["date"])
    .groupby("date", as_index=False)
    .agg(feeding_agg)
)

print("feeding_daily ->", feeding_daily.shape)
display(feeding_daily.head())


feeding_daily -> (271, 11)


,date,kg_am,kg_pm,kg_totales,sobrante,consumo,rechazo,kg_consumido_por_vaca,promedio_corral,sobrante_pct,consumo_pct
0,25-07-02,3500.0,3500.0,7000.0,265.0,6735.0,0.040370,NaN,NaN,0.040370,0.959630
1,25-07-03,3800.0,3900.0,7700.0,140.0,7560.0,0.020185,NaN,NaN,0.020185,0.979815
2,25-07-04,3750.0,3700.0,7450.0,375.0,7075.0,0.052725,NaN,NaN,0.052725,0.947275
3,25-07-05,3800.0,3800.0,7600.0,420.0,7180.0,0.049757,48.750000,NaN,0.059708,0.950243
4,25-07-06,4000.0,4000.0,8000.0,435.0,7565.0,0.056114,47.654609,47.654609,0.056114,0.943886


### 5.6 Dieta por día

La dieta es una variable de periodo, así que se expandirá a una tabla diaria por fecha.


In [32]:
diet_daily_frames = []

for _, row in diet_raw.iterrows():
    if pd.isna(row.get("period_start")) or pd.isna(row.get("period_end")):
        continue

    dates = pd.date_range(row["period_start"], row["period_end"], freq="D")
    tmp = pd.DataFrame({"date": dates})

    for c in diet_raw.columns:
        if c not in ["period_start", "period_end"]:
            tmp[c] = row[c]

    diet_daily_frames.append(tmp)

diet_daily = pd.concat(diet_daily_frames, ignore_index=True) if diet_daily_frames else pd.DataFrame(columns=["date"])

if not diet_daily.empty:
    diet_daily["date"] = pd.to_datetime(diet_daily["date"]).dt.normalize()

print("diet_daily ->", diet_daily.shape)
display(diet_daily.head())


diet_daily -> (273, 14)


,date,diet_period,n_ingredientes,diet_dm_total,diet_wet_total,diet_pct_ms_total,usa_oro_milk,usa_oro_balance,usa_silo_maiz,usa_silo_avena,usa_ensilado,usa_heno,usa_triticale,usa_melaza
0,2025-01-01,dieta Mayo,5,NaN,NaN,NaN,1,0,0,0,0,0,1,1
1,2025-01-02,dieta Mayo,5,NaN,NaN,NaN,1,0,0,0,0,0,1,1
2,2025-01-03,dieta Mayo,5,NaN,NaN,NaN,1,0,0,0,0,0,1,1
3,2025-01-04,dieta Mayo,5,NaN,NaN,NaN,1,0,0,0,0,0,1,1
4,2025-01-05,dieta Mayo,5,NaN,NaN,NaN,1,0,0,0,0,0,1,1


## 6. Diagnóstico rápido de llaves antes del merge


In [33]:
set_milk = set(zip(milking_daily["cow_id"], milking_daily["date"]))
set_rum = set(zip(rumination_daily["cow_id"], rumination_daily["date"]))
overlap = len(set_milk & set_rum)

print("milking_daily     ->", milking_daily.shape)
print("rumination_daily  ->", rumination_daily.shape)
print("weather_daily     ->", weather_daily.shape)
print("events_daily      ->", events_daily.shape)
print("feeding_daily     ->", feeding_daily.shape)
print("diet_daily        ->", diet_daily.shape)

print("\nOverlap rumia con ordeño (cow_id + date):", overlap)


milking_daily     -> (9814, 11)
rumination_daily  -> (7098, 8)
weather_daily     -> (1224, 3)
events_daily      -> (3822, 4)
feeding_daily     -> (271, 11)
diet_daily        -> (273, 14)

Overlap rumia con ordeño (cow_id + date): 28


### Comentarios y observaciones

Si el overlap de rumia con ordeño es muy pequeño, eso indica un problema de cobertura temporal o de definición de fecha entre ambas fuentes.


## 7. Merge principal


In [34]:
merged = milking_daily.copy()

# rumia
if not rumination_daily.empty:
    merged = merged.merge(
        rumination_daily,
        on=["cow_id", "date"],
        how="left",
        suffixes=("", "_rum")
    )

# clima
if not weather_daily.empty:
    merged = merged.merge(
        weather_daily,
        on="date",
        how="left"
    )

# eventos
if not events_daily.empty:
    merged = merged.merge(
        events_daily,
        on=["cow_id", "date"],
        how="left"
    )

# alimentación global por fecha
if not feeding_daily.empty:
    merged = merged.merge(
        feeding_daily,
        on="date",
        how="left",
        suffixes=("", "_feeding")
    )

# dieta por fecha
if not diet_daily.empty:
    merged = merged.merge(
        diet_daily,
        on="date",
        how="left",
        suffixes=("", "_diet")
    )

print("Shape merged completo:", merged.shape)
display(merged.head())


Shape merged completo: (9814, 44)


,cow_id,date,produccion_kg,di,dd,ti,td,ubre,destino_leche,ms,source_file,days_in_milk,lactation_age,weekday,month,group_id,source_file_rum,temperature_2m,relative_humidity_2m,eventos_pdf_count,eventos_pdf_text,kg_am,kg_pm,kg_totales,sobrante,consumo,rechazo,kg_consumido_por_vaca,promedio_corral,sobrante_pct,consumo_pct,diet_period,n_ingredientes,diet_dm_total,diet_wet_total,diet_pct_ms_total,usa_oro_milk,usa_oro_balance,usa_silo_maiz,usa_silo_avena,usa_ensilado,usa_heno,usa_triticale,usa_melaza
0,1204,2025-01-01,16.68,5.16,4.64,0.00,6.88,0,Tanque,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,NaN,NaN,14.266667,43.291667,<NA>,NaN,4000.0,4000.0,8000.0,10.0,7990.0,NaN,NaN,NaN,0.001282,0.998718,dieta Mayo,5,NaN,NaN,NaN,1,0,0,0,0,0,1,1
1,1204,2025-01-02,17.91,2.65,4.44,5.75,5.07,1,Divert 3,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,NaN,NaN,13.404167,64.916667,<NA>,NaN,3950.0,3950.0,7900.0,710.0,7190.0,NaN,NaN,NaN,0.095335,0.904665,dieta Mayo,5,NaN,NaN,NaN,1,0,0,0,0,0,1,1
2,1204,2025-01-03,25.36,6.26,6.52,2.77,9.81,1,Tanque,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,NaN,NaN,12.791667,73.000000,<NA>,NaN,3950.0,3900.0,7850.0,275.0,7575.0,NaN,NaN,NaN,0.035445,0.964555,dieta Mayo,5,NaN,NaN,NaN,1,0,0,0,0,0,1,1
3,1204,2025-01-04,16.71,4.13,3.89,2.96,5.73,1,Tanque,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,NaN,NaN,14.333333,70.750000,<NA>,NaN,3950.0,3950.0,7900.0,395.0,7505.0,NaN,NaN,NaN,0.051798,0.948202,dieta Mayo,5,NaN,NaN,NaN,1,0,0,0,0,0,1,1
4,1204,2025-01-05,25.04,5.85,5.94,4.85,8.40,1,Tanque,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,NaN,NaN,14.333333,58.458333,<NA>,NaN,3950.0,3950.0,7900.0,505.0,7395.0,NaN,NaN,NaN,0.066472,0.933528,dieta Mayo,5,NaN,NaN,NaN,1,0,0,0,0,0,1,1


## 8. Reporte de valores faltantes


In [35]:
missing_report = (
    pd.DataFrame({
        "column": merged.columns,
        "nan_count": merged.isna().sum().values,
        "nan_pct": (merged.isna().mean().values * 100)
    })
    .sort_values("nan_pct", ascending=False)
    .reset_index(drop=True)
)

display(missing_report)


,column,nan_count,nan_pct
0,eventos_pdf_text,9814,100.000000
1,eventos_pdf_count,9814,100.000000
2,diet_dm_total,9814,100.000000
3,diet_pct_ms_total,9814,100.000000
4,diet_wet_total,9814,100.000000
5,group_id,9786,99.714693
6,month,9786,99.714693
7,source_file_rum,9786,99.714693
8,days_in_milk,9786,99.714693
9,lactation_age,9786,99.714693


### Comentarios y observaciones

Este reporte ayuda a distinguir entre:

- columnas correctamente integradas
- columnas parcialmente disponibles
- columnas prácticamente vacías


## 9. Revisión de cobertura por mes


In [36]:
merged["mes_merge"] = merged["date"].dt.to_period("M")

print("Fechas únicas por mes en dataset final:")
display(merged.groupby("mes_merge")["date"].nunique())

print("Filas por mes en dataset final:")
display(merged.groupby("mes_merge").size())


Fechas únicas por mes en dataset final:


mes_merge
2025-01    31
2025-02    28
2025-03    31
2025-04    30
2025-05    31
2025-06    18
2025-08     1
2025-09    29
Freq: M, Name: date, dtype: int64

Filas por mes en dataset final:


mes_merge
2025-01    1757
2025-02    1649
2025-03    1809
2025-04    1748
2025-05    1789
2025-06    1024
2025-08       1
2025-09      37
Freq: M, dtype: int64

## 10. Guardado del dataset final


In [37]:
merged.head()

,cow_id,date,produccion_kg,di,dd,ti,td,ubre,destino_leche,ms,source_file,days_in_milk,lactation_age,weekday,month,group_id,source_file_rum,temperature_2m,relative_humidity_2m,eventos_pdf_count,eventos_pdf_text,kg_am,kg_pm,kg_totales,sobrante,consumo,rechazo,kg_consumido_por_vaca,promedio_corral,sobrante_pct,consumo_pct,diet_period,n_ingredientes,diet_dm_total,diet_wet_total,diet_pct_ms_total,usa_oro_milk,usa_oro_balance,usa_silo_maiz,usa_silo_avena,usa_ensilado,usa_heno,usa_triticale,usa_melaza,mes_merge
0,1204,2025-01-01,16.68,5.16,4.64,0.00,6.88,0,Tanque,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,NaN,NaN,14.266667,43.291667,<NA>,NaN,4000.0,4000.0,8000.0,10.0,7990.0,NaN,NaN,NaN,0.001282,0.998718,dieta Mayo,5,NaN,NaN,NaN,1,0,0,0,0,0,1,1,2025-01
1,1204,2025-01-02,17.91,2.65,4.44,5.75,5.07,1,Divert 3,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,NaN,NaN,13.404167,64.916667,<NA>,NaN,3950.0,3950.0,7900.0,710.0,7190.0,NaN,NaN,NaN,0.095335,0.904665,dieta Mayo,5,NaN,NaN,NaN,1,0,0,0,0,0,1,1,2025-01
2,1204,2025-01-03,25.36,6.26,6.52,2.77,9.81,1,Tanque,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,NaN,NaN,12.791667,73.000000,<NA>,NaN,3950.0,3900.0,7850.0,275.0,7575.0,NaN,NaN,NaN,0.035445,0.964555,dieta Mayo,5,NaN,NaN,NaN,1,0,0,0,0,0,1,1,2025-01
3,1204,2025-01-04,16.71,4.13,3.89,2.96,5.73,1,Tanque,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,NaN,NaN,14.333333,70.750000,<NA>,NaN,3950.0,3950.0,7900.0,395.0,7505.0,NaN,NaN,NaN,0.051798,0.948202,dieta Mayo,5,NaN,NaN,NaN,1,0,0,0,0,0,1,1,2025-01
4,1204,2025-01-05,25.04,5.85,5.94,4.85,8.40,1,Tanque,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,NaN,NaN,14.333333,58.458333,<NA>,NaN,3950.0,3950.0,7900.0,505.0,7395.0,NaN,NaN,NaN,0.066472,0.933528,dieta Mayo,5,NaN,NaN,NaN,1,0,0,0,0,0,1,1,2025-01


In [38]:
BASE_PROCESSED.mkdir(parents=True, exist_ok=True)

merged.to_parquet(OUT_TRAINING, index=False)

print("Archivo guardado en:")
print(OUT_TRAINING)


Archivo guardado en:
../data/processed/training_dataset.parquet


## 11. Conclusiones

En este notebook se integraron todas las fuentes intermedias disponibles:

- ordeño
- rumia
- clima
- eventos
- alimentación
- dieta

### Resultado principal

Se generó un dataset final a nivel:

```text
cow_id + date
```

con variables adicionales provenientes de otras fuentes.

### Nota importante sobre alimentación

Como todavía no existe un mapeo vaca-corral por fecha, la alimentación se integró de forma agregada a nivel diario global.  
Más adelante, si se obtiene ese mapeo, se podrá hacer un merge más preciso a nivel corral.

### Próximos pasos sugeridos

1. revisar cobertura real de rumia
2. revisar cobertura real del ordeño en julio/agosto/septiembre
3. integrar correctamente corral por vaca si se dispone
4. volver a correr EDA y feature engineering sobre el dataset final
